# Azure AI Agent Service Enterprise Demo

### Import Necessary Libraries
In this cell, we import all the libraries and modules required for the project.
This includes Azure AI SDKs, Gradio for UI, and custom functions.

```shell
az login --tenant <tenant id>
```

In [1]:
import os
import json
from typing import Any, List, Dict
from dotenv import load_dotenv

# (Optional) Gradio app for UI
import gradio as gr
from gradio import ChatMessage

# Azure AI Foundry (Projects v2) — agents are now part of azure-ai-projects.
# `azure-ai-agents` is no longer used in this notebook.
from azure.ai.projects import AIProjectClient
import azure.ai.projects as projectslib
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
    FileSearchTool,
    AzureAISearchTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
    FunctionTool,
)

# OpenAI types used to feed function-call results back to the agent
from openai.types.responses.response_input_param import (
    FunctionCallOutput,
    ResponseInputParam,
)

# Your custom Python functions (for "fetch_datetime", etc.)
from utils.enterprise_functions import fetch_datetime

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

Environment and authentication OK


### Create Client and Load Azure AI Foundry
Here, we initialize the Azure AI client using DefaultAzureCredential.
This allows us to authenticate and connect to the Azure AI service.

In [2]:
# AI Foundry Project endpoint (new Foundry Projects v2 API).
# Format: https://<account>.services.ai.azure.com/api/projects/<project>
project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
)
openai_client = project_client.get_openai_client()

print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-projects version: {projectslib.__version__}")

project_client api version: v1
azure-ai-projects version: 2.1.0


### Set Up Tools (BingGroundingTool, FileSearchTool)
In this step, we configure tools such as `BingGroundingTool` and `FileSearchTool`.
We check for existing connections and create or reuse vector stores for document search.

Note:
If you see the following cell has error:
```
AzureCliCredential: Please run 'az login' to set up an account
```

relogin from powershell
```powershell
az logout
az account clear
az login --tenant 00000000-0000-0000-0000-000000000000
```


In [3]:
# --- Bing Grounding tool ---
# In Projects v2 the Bing tool needs the project connection ID (not name).
bing_tool = None
try:
    bing_connection = project_client.connections.get(name=os.environ["BING_CONNECTION_NAME"])
    bing_tool = BingGroundingTool(
        bing_grounding=BingGroundingSearchToolParameters(
            search_configurations=[
                BingGroundingSearchConfiguration(
                    project_connection_id=bing_connection.id,
                    count=7,        # number of search results
                    set_lang="en",
                    market="en-us",
                    # freshness="2026-01-01", # single day
                    # freshness="2025-01-01..2026-05-04", # date range (also supported)
                    freshness="Month", # last 30 days only, or "Week"
                )
            ]
        )
    )
    print("bing > connected")
except Exception as e:
    bing_tool = None
    print(f"bing failed > no connection found or permission issue ({e})")

# --- File Search tool: vector store now lives on the OpenAI client ---
FOLDER_NAME = "enterprise-data"
VECTOR_STORE_NAME = "hr-policy-vector-store"

all_vector_stores = list(openai_client.vector_stores.list())
existing_vector_store = next(
    (store for store in all_vector_stores if store.name == VECTOR_STORE_NAME),
    None,
)

vector_store_id = None
if existing_vector_store:
    vector_store_id = existing_vector_store.id
    print(f"reusing vector store > {existing_vector_store.name} (id: {existing_vector_store.id})")
else:
    if os.path.isdir(FOLDER_NAME):
        vector_store = openai_client.vector_stores.create(name=VECTOR_STORE_NAME)
        vector_store_id = vector_store.id
        print(f"created vector store > {vector_store.name} (id: {vector_store_id})")

        for file_name in os.listdir(FOLDER_NAME):
            file_path = os.path.join(FOLDER_NAME, file_name)
            if os.path.isfile(file_path):
                print(f"uploading > {file_name}")
                with open(file_path, "rb") as f:
                    openai_client.vector_stores.files.upload_and_poll(
                        vector_store_id=vector_store_id,
                        file=f,
                    )

file_search_tool = None
if vector_store_id:
    file_search_tool = FileSearchTool(vector_store_ids=[vector_store_id])

bing > connected
reusing vector store > hr-policy-vector-store (id: vs_AMBfvCNoWCerPKW15891APxf)


### Setup AI Search tool (Azure AI Search)

In [4]:
# Get the project connection ID for your Azure AI Search resource
search_tool = None
try:
    aisearch_connections = project_client.connections.list()
    idx_conn_id = next(
        c.id for c in aisearch_connections if c.name == os.environ.get("AZURE_SEARCH_CONNECTION_NAME")
    )

    search_tool = AzureAISearchTool(
        azure_ai_search=AzureAISearchToolResource(
            indexes=[
                AISearchIndexResource(
                    project_connection_id=idx_conn_id,
                    index_name=os.environ.get("AZURE_SEARCH_INDEX_NAME"),
                    query_type=AzureAISearchQueryType.SIMPLE,
                ),
            ]
        )
    )
    print("azure ai search > connected directly to index")
except Exception as e:
    search_tool = None
    print(f"azure ai search > skipped (no connection configured): {str(e)}")


azure ai search > connected directly to index


### Combine All Tools into a List
In Projects v2, agents take a plain `tools=[...]` list — there's no `ToolSet`
and no `enable_auto_function_calls`. Function-call execution happens explicitly
in the Responses loop (see the chat handler below).

In [5]:
# --- Declarative function tool: fetch_datetime ---
# In v2, function tools are declared by JSON Schema. The model emits
# `function_call` items in the response stream; we execute them and feed the
# result back via `FunctionCallOutput`.
fetch_datetime_tool = FunctionTool(
    name="fetch_datetime",
    description="Returns the current UTC date/time as JSON.",
    parameters={
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
    strict=True,
)

# Map function name -> Python callable so the chat loop can dispatch.
function_dispatch: Dict[str, Any] = {
    "fetch_datetime": lambda **_: fetch_datetime(),
}

# --- Build the tools list in the order the agent should consider them ---
tools: List[Any] = []
if bing_tool:
    tools.append(bing_tool)
if file_search_tool:
    tools.append(file_search_tool)
if search_tool:
    tools.append(search_tool)
tools.append(fetch_datetime_tool)

for t in tools:
    print(f"tool > {type(t).__name__} (type={getattr(t, 'type', '?')})")

tool > BingGroundingTool (type=bing_grounding)
tool > FileSearchTool (type=file_search)
tool > AzureAISearchTool (type=azure_ai_search)
tool > FunctionTool (type=function)


### Create or Update the Enterprise Agent (versioned)
In Projects v2, agents are versioned resources. Each call to `create_version`
either creates the agent (if it doesn't exist) or pins a new version of an
existing agent. There is no in-place `update_agent` — just create a new version
with the desired model, instructions, and tools.

In [6]:
AGENT_NAME = "enterprise-knowledge-agent"
model_name = settings.model_deployment_name

# Tool name conventions in v2 stream events:
#   azure_ai_search   🔎 ai search private index
#   file_search       📄 searching docs
#   bing_grounding    🔍 searching bing
#   function_call     🕠 fetch_datetime, ...

instructions = (
    "You are a helpful enterprise assistant at Contoso.\n\n"
    "## CRITICAL: Call exactly ONE tool per question. Never call multiple tools.\n\n"
    "## Tool selection (pick the single best match):\n"
    " * azure_ai_search \u2014 ANY question about financial / fiscal / annual "
    "reports, earnings, revenue, balance sheet, or company-indexed documents. "
    "This is the ONLY tool for fiscal reports. NEVER use file_search for "
    "financial topics.\n"
    " * file_search \u2014 ONLY for HR documents: HR policy, code of conduct, "
    "holiday/vacation, remote work, performance review. Nothing else.\n"
    " * bing_grounding \u2014 current public news, web search.\n"
    " * fetch_datetime \u2014 current date/time.\n"
    "\n"
    "## Disambiguation rules:\n"
    " * 'fiscal' / 'financial' / 'report' / 'earnings' \u2192 azure_ai_search (NEVER file_search)\n"
    " * 'latest news' / 'public news' / 'news about' \u2192 bing_grounding\n"
    " * 'company source' + financial/fiscal topic \u2192 azure_ai_search\n"
    " * 'company source' + HR/policy topic \u2192 file_search\n"
    "\n"
    "## Guidelines:\n"
    "Provide well-structured and professional answers. Always return the "
    "grounding links for citations."
)

agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=instructions,
        tools=tools,
    ),
    description="Enterprise knowledge agent with Bing, File Search, AI Search, and a datetime function.",
)
print(f"agent > {agent.name} (id: {agent.id}, version: {agent.version})")

agent > enterprise-knowledge-agent (id: enterprise-knowledge-agent:7, version: 7)


### Create a Conversation
In Projects v2, multi-turn state is held by an OpenAI **conversation**
(via `openai_client.conversations.create()`). It replaces the old `thread`
concept and is what `responses.create(conversation=...)` reads from / writes to.

In [7]:
conversation = openai_client.conversations.create()
print(f"conversation > created (id: {conversation.id})")

conversation > created (id: conv_864d67e3ca25c4e200HhlRsXHS7I8r8WUdYSI16JVT7EWYyzcZ)


## (Optional) Quick test helpers

A tiny helper that calls the agent through `responses.create` and iteratescells below.
function-calls until the model produces a final message. Used by the test

In [8]:
AGENT_REFERENCE = {"name": agent.name, "type": "agent_reference"}


def run_agent_once(user_prompt: str, conversation_id: str | None = None, tool_choice="auto"):
    """Send `user_prompt` to the agent and resolve any function calls in a loop.

    Returns the final Responses object once the model produced a message
    output (i.e. no pending function_call items).
    """
    kwargs: Dict[str, Any] = {
        "input": user_prompt,
        "extra_body": {"agent_reference": AGENT_REFERENCE},
        "tool_choice": tool_choice,
    }
    if conversation_id:
        kwargs["conversation"] = conversation_id

    response = openai_client.responses.create(**kwargs)

    while True:
        pending = [item for item in response.output if item.type == "function_call"]
        if not pending:
            return response

        input_list: ResponseInputParam = []
        for item in pending:
            fn = function_dispatch.get(item.name)
            if fn is None:
                output = json.dumps({"error": f"unknown function {item.name}"})
            else:
                args = json.loads(item.arguments) if item.arguments else {}
                print(f"{item.name} inputs > {args} (call_id:{item.call_id})")
                output = fn(**args)
                print(f"output > {output}")
            input_list.append(FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=output,
            ))

        response = openai_client.responses.create(
            input=input_list,
            previous_response_id=response.id,
            extra_body={"agent_reference": AGENT_REFERENCE},
        )


def display_response(response) -> None:
    """Print the final agent text plus any URL/file citation annotations."""
    print(f"agent > {response.output_text}\n")
    for item in response.output:
        if item.type != "message":
            continue
        for content in item.content:
            if getattr(content, "type", None) != "output_text":
                continue
            for ann in getattr(content, "annotations", []) or []:
                if ann.type == "url_citation":
                    print(f"  url > {getattr(ann, 'title', '')} {ann.url}")
                elif ann.type == "file_citation":
                    print(f"  file > {getattr(ann, 'filename', '')} (id: {ann.file_id})")


In [9]:
# --- Testing AI Search tool ---
# `tool_choice="required"` forces the agent to call a tool. The Foundry
# Responses endpoint accepts only the strings "auto" / "none" / "required",
# or `{"type": "function", "name": "..."}` for a specific function tool.
# Hosted tools (azure_ai_search, bing_grounding, file_search) cannot be
# pinned by name; rely on the agent instructions + "required" to select them.
ai_search_conv = openai_client.conversations.create()
print(f"conversation > created (id: {ai_search_conv.id})")

user_prompt_1 = "summarize siemens fiscal report 2024 from my company source"
resp_1 = run_agent_once(
    user_prompt_1,
    conversation_id=ai_search_conv.id,
    tool_choice="required",
)

conversation > created (id: conv_32c43efba47f3c5f0061qCYvSmxiakyYbwh8gJ6Iv7jlQVfOR2)


In [10]:
display_response(resp_1)

agent > The Siemens fiscal report for 2024 highlights a strong financial performance with key results including:

- Net income reached a historic high of €9.0 billion.
- Basic earnings per share (EPS) increased to €10.53, with EPS pre-PPA at €11.15 (or €10.54 excluding Siemens Energy Investment).
- Return on capital employed (ROCE) rose to 19.1%, within the target range of 15% to 20%.
- The capital structure remained strong with an industrial net debt to EBITDA ratio of 0.7.
- Free cash flow was excellent at €9.5 billion, slightly below the record €10.0 billion in 2023.
- Profit margins improved in several segments: Digital Industries and Mobility had strong margins, though Digital Industries saw a decline to 18.9%.
- Siemens Financial Services showed significant earnings growth with a return on equity after tax of 17.6%.
- The company maintained its guidance and forecasts for EPS, ROCE, and capital structure.

The report also acknowledges ongoing global economic uncertainties includin

### Testing Bing grounding tool

In [11]:
# --- Testing Bing grounding tool ---
bing_conv = openai_client.conversations.create()
print(f"conversation > created (id: {bing_conv.id})")

user_prompt_2 = "tell me about the latest news about microsoft azure"
resp_2 = run_agent_once(user_prompt_2, conversation_id=bing_conv.id)

conversation > created (id: conv_d11580b107dd20c300L2qjRVn8vomMW6nuFHG6g959Y6DtNg32)


In [12]:
display_response(resp_2)

agent > The latest news about Microsoft Azure highlights strong growth and advancements especially driven by artificial intelligence (AI) capabilities:

1. Microsoft reported an impressive 40% growth in Azure cloud revenue in the fiscal third quarter of 2026, exceeding both company guidance and analyst expectations. This growth is partly due to new data center capacity and surging demand for AI-powered workloads through Azure OpenAI Service and Copilot integrations.

2. Microsoft's AI business has reached a $37 billion annual revenue run rate, more than doubling from a year ago, indicating that AI workloads have shifted from pilot projects to sustained, billable cloud consumption.

3. Microsoft's overall cloud revenue, including Azure, Microsoft 365, LinkedIn, and Dynamics 365, grew 29% year-over-year to $54.5 billion.

4. The company is investing heavily in infrastructure, with a forecast of $190 billion in capital expenditures for 2026 due to increasing memory prices and the need for

### Add Guardrails with Blocklist

Visiable at Build -> Guardrails -> Guardails and Blocklists


In [13]:
## Add a block list to filter 
# put code here

from utils.blocklist_helper import setup_blocklist_policy

setup_blocklist_policy(
    credential=credential,
    project_endpoint=settings.project_endpoint,
    deployment_name=settings.model_deployment_name,
    blocklist_name="competitor-cloud", # Blocklist entry name in Foundry project
    policy_name="strict-with-competitor-blocklist", # Guardrail policy entry name in Foundry project
    description="Block competitor cloud mentions",
    # in your setup_blocklist_policy(...) call
    patterns=[
    # Google Cloud
        ("google", r"\bgoogle\b", True),
        ("gcp", r"\bgcp\b", True),
        ("google-cloud", r"\bgoogle\s+cloud(?:\s+platform)?\b", True),
        ("vertex-ai", r"\bvertex\s+ai\b", True),
        ("bigquery", r"\bbigquery\b", True),
        ("cloud-run", r"\bcloud\s+run\b", True),
        ("anthos", r"\banthos\b", True),
        # AWS
        ("aws", r"\baws\b", True),
        ("amazon-web-services", r"\bamazon\s+web\s+services\b", True),
        ("ec2", r"\bec2\b", True),
        ("s3", r"\bs3\b", True),
        ("lambda-aws", r"\baws\s+lambda\b", True),
        ("dynamodb", r"\bdynamodb\b", True),
        ("sagemaker", r"\bsagemaker\b", True),
        ("bedrock", r"\bbedrock\b", True),
    ],
    override=False,   # so the existing items get re-uploaded
)

resolved > sub=6753****-****-.... rg=rg-ai-sandbox-yw-uno account=foundry-proj-yw-uno-resource deployment=gpt-4.1-mini
blocklist > competitor-cloud already exists (skipping; pass override=True to recreate)
policy > strict-with-competitor-blocklist already exists (skipping; pass override=True to recreate)
deployment > gpt-4.1-mini now uses raiPolicyName=strict-with-competitor-blocklist


In [14]:
# --- Testing the Content Filter with Block List ---
block_list_conv = openai_client.conversations.create()
print(f"conversation > created (id: {block_list_conv.id})")

user_prompt_3 = "what is the latest news about google gcp?"
resp_3 = run_agent_once(user_prompt_3, conversation_id=block_list_conv.id)

conversation > created (id: conv_52a985470b1ac37300PZ9YBU6MS0N2jI3b4Rst3aLqqpRnjV81)


In [15]:
display_response(resp_3)

agent > The latest news about Google Cloud Platform (GCP) highlights several key developments and announcements from Google Cloud Next 2026:

1. Google Cloud is emphasizing the transition to an "agenticI'm sorry, but I cannot assist with that request.



### Streaming with the Responses API
Projects v2 streams an OpenAI **Responses** event sequence (no `AgentEventHandler`
subclass). The chat function below iterates `event.type` directly and handles:

and continue streaming.

* `response.created` — a new response startedcallable, send a `FunctionCallOutput` back via a follow-up `responses.create`,

* `response.output_item.added` / `.done` — a tool call (`bing_grounding`,When the model emits a `function_call` item, we execute the matching Python

  `file_search`, `azure_ai_search`, `code_interpreter`, `function_call`, ...)

  starts/finishes* `response.completed` — the response is finished
* `response.output_text.delta` — streamed assistant text

In [16]:
# UI labels for tool-call bubbles in the Gradio chat.
FUNCTION_TITLES = {
    "fetch_datetime": "🕒 fetching datetime",
    "file_search": "📄 searching docs",
    "bing_grounding": "🔍 searching bing",
    "azure_ai_search": "🔎 ai search private index",
    "code_interpreter": "📊 analyzing data",
}

# Map a streamed output-item type to a function-title key.
TOOL_ITEM_TO_KEY = {
    "bing_grounding_call": "bing_grounding",
    "file_search_call": "file_search",
    "azure_ai_search_call": "azure_ai_search",
    "code_interpreter_call": "code_interpreter",
}

def _tool_title(name: str) -> str:
    return FUNCTION_TITLES.get(name, f"🛠 calling {name}")

### Implement the Main Chat Functions
These functions define how user messages and tool interactions are processed.
It uses the agent's thread to handle conversations and streams partial responses.

In [17]:
def convert_dict_to_chatmessage(msg: dict) -> ChatMessage:
    """Convert a legacy dict-based history entry to a gr.ChatMessage."""
    return ChatMessage(
        role=msg["role"],
        content=msg["content"],
        metadata=msg.get("metadata", None),
    )


def _format_annotation(ann) -> str:
    """Render a Responses annotation as a Markdown link."""
    a_type = getattr(ann, "type", None)
    if a_type == "url_citation":
        title = getattr(ann, "title", "") or "source"
        return f" [{title}]({ann.url})"
    if a_type == "file_citation":
        filename = getattr(ann, "filename", "") or "file"
        return f" [{filename}]"
    return ""


def _stream_response(resp_iter, conversation, in_progress_tools: Dict[str, ChatMessage]):
    """Consume one streamed Responses iterator.

    Yields `(conversation, "")` after each visible state change so Gradio can
    repaint. Returns a list of pending function calls that the caller must
    execute and feed back via a follow-up `responses.create`.
    """
    pending_function_calls: List[Any] = []
    last_response_id: str | None = None
    assistant_msg: ChatMessage | None = None

    for event in resp_iter:
        etype = getattr(event, "type", "")

        if etype == "response.created":
            last_response_id = event.response.id

        elif etype == "response.output_item.added":
            item = event.item
            item_type = getattr(item, "type", "")

            if item_type == "message":
                # Start a fresh assistant bubble
                assistant_msg = ChatMessage(
                    role="assistant",
                    content="",
                    metadata={"id": item.id},
                )
                conversation.append(assistant_msg)
                yield conversation, ""

            elif item_type == "function_call":
                # Show a pending bubble for the function call
                bubble = ChatMessage(
                    role="assistant",
                    content="",
                    metadata={
                        "title": _tool_title(item.name),
                        "status": "pending",
                        "id": f"tool-{item.call_id}",
                    },
                )
                conversation.append(bubble)
                in_progress_tools[item.call_id] = bubble
                yield conversation, ""

            elif item_type in TOOL_ITEM_TO_KEY:
                key = TOOL_ITEM_TO_KEY[item_type]
                bubble = ChatMessage(
                    role="assistant",
                    content="searching...",
                    metadata={
                        "title": _tool_title(key),
                        "status": "pending",
                        "id": f"tool-{item.id}",
                    },
                )
                conversation.append(bubble)
                in_progress_tools[item.id] = bubble
                yield conversation, ""

        elif etype == "response.function_call_arguments.delta":
            bubble = in_progress_tools.get(getattr(event, "call_id", ""))
            if bubble is not None:
                bubble.content = (bubble.content or "") + event.delta
                yield conversation, ""

        elif etype == "response.output_item.done":
            item = event.item
            item_type = getattr(item, "type", "")
            if item_type == "function_call":
                pending_function_calls.append(item)
                bubble = in_progress_tools.pop(item.call_id, None)
                if bubble is not None:
                    bubble.metadata["status"] = "done"
                    bubble.content = item.arguments or ""
                yield conversation, ""
            elif item_type in TOOL_ITEM_TO_KEY:
                bubble = in_progress_tools.pop(item.id, None)
                if bubble is not None:
                    bubble.metadata["status"] = "done"
                yield conversation, ""
            elif item_type == "message":
                # Decorate the final message with citation links
                if assistant_msg is not None:
                    for content in getattr(item, "content", []) or []:
                        if getattr(content, "type", None) != "output_text":
                            continue
                        for ann in getattr(content, "annotations", []) or []:
                            assistant_msg.content += _format_annotation(ann)
                assistant_msg = None
                yield conversation, ""

        elif etype == "response.output_text.delta":
            if assistant_msg is None:
                assistant_msg = ChatMessage(role="assistant", content="")
                conversation.append(assistant_msg)
            assistant_msg.content = (assistant_msg.content or "") + event.delta
            yield conversation, ""

        elif etype == "response.completed":
            yield conversation, ""

        elif etype == "response.failed":
            err = getattr(event.response, "error", None)
            conversation.append(ChatMessage(
                role="assistant",
                content=f"error > {err}",
            ))
            yield conversation, ""

    # Stash the response id on the iterator for the caller
    resp_iter._last_response_id = last_response_id  # type: ignore[attr-defined]
    resp_iter._pending_calls = pending_function_calls  # type: ignore[attr-defined]


def azure_enterprise_chat(user_message: str, history: List[dict]):
    """Stream a multi-turn agent reply through the Responses API.

    Iteratively resolves any function calls until the agent produces a final
    message. Uses the global `conversation` for multi-turn state.
    """
    # Convert existing history from dict to ChatMessage
    chat: List[ChatMessage] = [convert_dict_to_chatmessage(m) for m in history]
    chat.append(ChatMessage(role="user", content=user_message))
    yield chat, ""

    in_progress_tools: Dict[str, ChatMessage] = {}

    # First request: send the user prompt
    stream = openai_client.responses.create(
        stream=True,
        conversation=conversation.id,
        input=user_message,
        extra_body={"agent_reference": AGENT_REFERENCE},
    )

    while True:
        # Drain this stream, yielding to Gradio along the way
        for snapshot in _stream_response(stream, chat, in_progress_tools):
            yield snapshot

        pending = getattr(stream, "_pending_calls", []) or []
        last_id = getattr(stream, "_last_response_id", None)
        if not pending or last_id is None:
            return

        # Execute every pending function call and feed results back
        outputs: ResponseInputParam = []
        for item in pending:
            fn = function_dispatch.get(item.name)
            if fn is None:
                output = json.dumps({"error": f"unknown function {item.name}"})
            else:
                args = json.loads(item.arguments) if item.arguments else {}
                output = fn(**args)
            outputs.append(FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=output,
            ))

        stream = openai_client.responses.create(
            stream=True,
            input=outputs,
            previous_response_id=last_id,
            extra_body={"agent_reference": AGENT_REFERENCE},
        )

### Build a Gradio UI
Create a Gradio interface for interacting with the enterprise agent. 
Include a chatbot component and a text input box for user queries.

In [18]:
brand_theme = gr.themes.Default(
    primary_hue="blue",
    secondary_hue="blue",
    neutral_hue="gray",
    font=["Segoe UI", "Arial", "sans-serif"],
    font_mono=["Courier New", "monospace"],
    text_size="lg",
).set(
    button_primary_background_fill="#0f6cbd",
    button_primary_background_fill_hover="#115ea3",
    button_primary_background_fill_hover_dark="#4f52b2",
    button_primary_background_fill_dark="#5b5fc7",
    button_primary_text_color="#ffffff",
    button_secondary_background_fill="#e0e0e0",
    button_secondary_background_fill_hover="#c0c0c0",
    button_secondary_background_fill_hover_dark="#a0a0a0",
    button_secondary_text_color="#000000",
    body_background_fill="#f5f5f5",
    block_background_fill="#ffffff",
    body_text_color="#242424",
    body_text_color_subdued="#616161",
    block_border_color="#d1d1d1",
    block_border_color_dark="#333333",
    input_background_fill="#ffffff",
    input_border_color="#d1d1d1",
    input_border_color_focus="#0f6cbd",
)

with gr.Blocks(theme=brand_theme, css="footer {visibility: hidden;}", fill_height=True) as demo:

    def clear_conversation():
        global conversation
        conversation = openai_client.conversations.create()
        return []

    def on_example_clicked(evt: gr.SelectData):
        return evt.value["text"]  # Fill the textbox with that example text

    gr.HTML("<h1 style=\"text-align: center;\">Azure AI Agent Service</h1>")

    chatbot = gr.Chatbot(
        examples=[
            {"text": "What's my company's remote work policy?"},
            # {"text": "Check if it will rain tomorrow?"},
            # {"text": "How is Contoso's stock doing today?"},
            # {"text": "Show a summary of the all HR policy."},
            # {"text": "What is the date today?"},
            {"text": "What is the latest news about Microsoft?"},
            {"text": "What is the latest public news about Microsoft Discovery Platform?"},
            # {"text": "what is the latest approach of microsoft to do quantum computing?"},
            {"text": "summarize siemens fiscal report 2024 from my company source"},
            # {"text": "Hight 5 insight regarding Microsoft Fiscal Report 2025 Q3."},
        ],
        show_label=False,
        scale=1,
    )

    textbox = gr.Textbox(
        show_label=False,
        lines=1,
        submit_btn=True,
    )

    # Populate textbox when an example is clicked
    chatbot.example_select(fn=on_example_clicked, inputs=None, outputs=textbox)

    # On submit: call azure_enterprise_chat, then clear the textbox
    (textbox
     .submit(
         fn=azure_enterprise_chat,
         inputs=[textbox, chatbot],
         outputs=[chatbot, textbox],
     )
     .then(
         fn=lambda: "",
         outputs=textbox,
     )
    )

    # A "Clear" button that resets the conversation and the Chatbot
    chatbot.clear(fn=clear_conversation, outputs=chatbot)

# Launch your Gradio app
if __name__ == "__main__":
    print("Note:\ntool calling may fail, close the chat with trash bin icon (🗑️) and rerun the prompt.\n")
    demo.launch()

/var/folders/x6/_vk8l9r96m75w07cmf8y05n80000gn/T/ipykernel_58344/473487225.py:29: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=brand_theme, css="footer {visibility: hidden;}", fill_height=True) as demo:


Note:
tool calling may fail, close the chat with trash bin icon (🗑️) and rerun the prompt.

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### (Optional) delete agent version, conversation, and vector store resources
Uncomment the next cell to delete the resources created in this notebook.

In Projects v2, agents are versioned: `delete_version` removes a single version,`delete` removes the agent and all its versions.

In [19]:
# from azure.identity import DefaultAzureCredential
# from azure.ai.projects import AIProjectClient
# import os

# credential = DefaultAzureCredential()
# project_client_delete = AIProjectClient(
#    credential=credential,
#    endpoint=os.environ.get("PROJECT_ENDPOINT")
# )
# openai_client_delete = project_client_delete.get_openai_client()

# try:
#    project_client_delete.agents.delete_version(
#        agent_name=agent.name,
#        agent_version=agent.version,
#    )
#    print("Agent version deletion successful.")
#    openai_client_delete.conversations.delete(conversation.id)
#    print("Conversation deletion successful.")
#    openai_client_delete.vector_stores.delete(vector_store_id)

#    print("Vector store deletion successful.")#    print(f"Error during deletion: {e}")

#    print("All deletions succeeded.")# except Exception as e:

In [20]:
# existing_stores = openai_client.vector_stores.list()
# for store in existing_stores:
#     openai_client.vector_stores.delete(store.id)